# Batch 08 · Exp04 · Validasi silang V3 & 40 episode irisan

Notebook mandiri hasil `shared/build_notebooks.py`. Jalankan **Run all** pada kernel Python dengan scikit-learn 1.7.2 (lihat `models/batch-08/requirements-lock.txt`).

In [1]:
import sys
from pathlib import Path

if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/Penelitian-Adaptive-IDE')
    DATASET_ROOT = PROJECT_ROOT / 'Dataset'
else:
    cwd = Path.cwd().resolve()
    PROJECT_ROOT = next((p for p in [cwd, *cwd.parents] if (p / 'analysis_sessions/dataset_fusion/output').exists()), cwd)
    DATASET_ROOT = PROJECT_ROOT / 'analysis_sessions/dataset_fusion/output'
BATCH_ROOT = PROJECT_ROOT / 'models/batch-08'
OUTPUT_ROOT = BATCH_ROOT / 'exp04_v3_crosscheck/outputs'
print('PROJECT_ROOT:', PROJECT_ROOT)
print('DATASET_ROOT:', DATASET_ROOT)


PROJECT_ROOT: D:\Penelitian Pak Hadi\Adaptive-IDE
DATASET_ROOT: D:\Penelitian Pak Hadi\Adaptive-IDE\analysis_sessions\dataset_fusion\output


## Kode bersama · `shared/b8_common.py`

In [2]:
"""Batch 08 · komponen bersama: data V4/V2, label, transformasi, model, CV, metrik.

Sebagian fungsi disalin dari `models/batch-07/exp03/experiment.py` agar referensi batch 07
dapat direproduksi persis (ditandai "asal: batch-07 exp03"). File ini diinline ke notebook
oleh `build_notebook.py` setiap eksperimen, dan dapat juga diimpor sebagai modul biasa.
"""

import hashlib
import json
import platform
import shutil
import sys
from datetime import datetime, timezone
from pathlib import Path
from uuid import uuid4

import numpy as np
import pandas as pd
import scipy
import sklearn
from scipy.special import expit, logit
from scipy.stats import chi2, spearmanr
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import average_precision_score, roc_auc_score

b8_common_loaded = True
SEED = 42

# --------------------------------------------------------------------------------------
# Fitur (definisi batch 07)
# --------------------------------------------------------------------------------------
HRV2 = ["rmssd_ms", "sdnn_ms"]
EYE7 = ["blink_rate_per_minute", "ear_left_mean", "ear_left_std", "ear_right_mean", "ear_right_std",
        "iris_size_estimate_mean", "iris_size_estimate_std"]
CORE9 = HRV2 + EYE7
HEAD4 = ["head_pitch_mean", "head_pitch_std", "head_yaw_mean", "head_yaw_std"]
BASE1 = ["rmssd_baseline_ratio"]
FEATURE_SETS = {
    "core9": CORE9,
    "core9_head4": CORE9 + HEAD4,
    "core9_head4_base": CORE9 + HEAD4 + BASE1,
}
HISTORY_FEATURES = CORE9 + HEAD4  # fitur yang punya median riwayat kausal
ELAPSED = "log_elapsed_active_s"

# Hash input yang sama dengan batch 07 (manifest exp03); v2 dicek dengan hash LF.
V2_ALL_LF_HASH = "acbef9a60f4114de460ee3c57ca94ff5cae0d48b67229d8b14283d173fef1355"

THRESHOLD_GRID = tuple(np.round(np.linspace(0.20, 0.80, 25), 3))
POLICIES = ("fixed_050", "youden", "recall75_min_fa", "fa_cap25_max_rec")
PRIMARY_POLICY = "youden"

# Label kebijakan resmi Batch 08
V4_LABEL_POLICY = "v4_overload_rating_ge7_v1"
V3_LABEL_POLICY = "v3_high_vs_rest_foldwise_v1"


# --------------------------------------------------------------------------------------
# Util
# --------------------------------------------------------------------------------------
def sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


def lf_hash(path):
    return hashlib.sha256(Path(path).read_bytes().replace(b"\r\n", b"\n")).hexdigest()


def save_json(path, value):
    Path(path).write_text(json.dumps(value, indent=2, ensure_ascii=False, allow_nan=False, default=str),
                          encoding="utf-8")


def new_run_dir(output_root, prefix):
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    out = Path(output_root) / f"{prefix}_{stamp}_{uuid4().hex[:8]}"
    out.mkdir(parents=True, exist_ok=False)
    return out


def environment_info():
    return {"python": sys.version.split()[0], "platform": platform.platform(), "numpy": np.__version__,
            "pandas": pd.__version__, "scipy": scipy.__version__, "sklearn": sklearn.__version__}


def zip_run(out):
    return Path(shutil.make_archive(str(out), "zip", root_dir=out.parent, base_dir=out.name))


def find_project_root(start=None):
    cwd = Path(start or Path.cwd()).resolve()
    for p in [cwd, *cwd.parents]:
        if (p / "analysis_sessions/dataset_fusion/output").exists():
            return p
    raise FileNotFoundError("Root proyek tidak ditemukan.")


def latest_run(directory, prefix):
    runs = sorted(p for p in Path(directory).glob(prefix + "_*") if p.is_dir())
    if not runs:
        raise FileNotFoundError(f"Belum ada run {prefix} di {directory}")
    return runs[-1]


# --------------------------------------------------------------------------------------
# Data V4
# --------------------------------------------------------------------------------------
def elapsed_at(intervals, t_ms):
    """Waktu aktif episode (detik) yang sudah berjalan sampai t_ms."""
    total = 0.0
    for start, end in intervals:
        if t_ms <= start:
            break
        total += min(end, t_ms) - start
    return total / 1000.0


def load_v4(dataset_root, rating_threshold=7):
    """Window V4 good_acceptable 15 s + metadata episode. Urutan baris mengikuti batch 07 exp03."""
    root = Path(dataset_root)
    v4 = root / "v4_self_reported_rating"
    hashes = {}
    for rel in ("window_15s_step_5s/good_acceptable.csv", "window_15s_step_5s/window_assignments.csv",
                "episode_ratings.csv"):
        hashes[f"v4/{rel}"] = sha256(v4 / rel)
    v2_all = root / "v2_session_phase/window_15s_step_5s/all_data.csv"
    if lf_hash(v2_all) != V2_ALL_LF_HASH:
        raise ValueError("Snapshot v2 all_data 15 s berbeda dari batch 07.")
    hashes["v2/window_15s_step_5s/all_data.csv"] = sha256(v2_all)

    df = pd.read_csv(v4 / "window_15s_step_5s/good_acceptable.csv")
    wa = pd.read_csv(v4 / "window_15s_step_5s/window_assignments.csv")
    ep = pd.read_csv(v4 / "episode_ratings.csv")
    df = df.merge(wa[["window_id", "episode_id", "difficulty_rating"]].rename(columns={"difficulty_rating": "wa"}),
                  on="window_id", how="left", validate="1:1")
    if not (df.difficulty_rating == df.wa).all():
        raise ValueError("difficulty_rating berbeda antara CSV dan window_assignments.")
    df = df.drop(columns="wa")
    df["window_start_dt"] = pd.to_datetime(df.window_start, utc=True, format="mixed")
    df["window_end_dt"] = pd.to_datetime(df.window_end, utc=True, format="mixed")
    df = df.sort_values(["session_id", "window_start_dt", "window_id"]).reset_index(drop=True)
    df["y"] = (df.difficulty_rating >= rating_threshold).astype(int)

    ep = ep.set_index("episode_id")
    intervals = ep.intervals.map(json.loads)
    end_ms = (df.window_end_dt - pd.Timestamp("1970-01-01", tz="UTC")) / pd.Timedelta(milliseconds=1)
    df["elapsed_active_s"] = [elapsed_at(intervals[e], t) for e, t in zip(df.episode_id, end_ms)]
    if (df.elapsed_active_s <= 0).any():
        raise ValueError("Window tanpa waktu aktif.")
    df[ELAPSED] = np.log(df.elapsed_active_s)

    session_start = ep.groupby("session_id").start_ms.min()
    meta = df.groupby("episode_id").agg(participant_id=("participant_id", "first"),
                                        session_id=("session_id", "first"),
                                        difficulty_rating=("difficulty_rating", "first"), y=("y", "first"),
                                        n_windows=("window_id", "count"))
    meta = meta.join(ep[["start_ms", "active_seconds", "puzzle_set", "timeout_submissions", "manual_errors"]])
    meta["minutes_into_session"] = (meta.start_ms - meta.session_id.map(session_start)) / 60000.0
    meta["difficulty_code"] = meta.puzzle_set.str.extract(r"^(easy|demo|medium|hard)")[0].map(
        {"easy": 0, "demo": 0, "medium": 1, "hard": 2})
    meta["label_policy"] = V4_LABEL_POLICY if rating_threshold == 7 else f"v4_overload_rating_ge{rating_threshold}_v1"
    return df, meta, hashes


def history_reference(dataset_root, df, meta):
    """Median riwayat kausal per episode (asal: batch-07 exp03 `prior_ref_median`).

    Maksimal 12 window disjoint all_data v2 (kualitas good/acceptable) yang berakhir sebelum episode dimulai,
    minimal 3. Hanya memakai waktu sebelum episode, sehingga tersedia real-time.
    """
    v2_all = Path(dataset_root) / "v2_session_phase/window_15s_step_5s/all_data.csv"
    keys, times = ["participant_id", "session_id"], ["window_start", "window_end"]
    bank = pd.read_csv(v2_all, usecols=keys + times + ["quality_tier"] + HISTORY_FEATURES)
    bank = bank[bank.quality_tier.isin(["good", "acceptable"]) & np.isfinite(bank[HISTORY_FEATURES]).all(axis=1)].copy()
    for f in times:
        bank[f] = pd.to_datetime(bank[f], utc=True, format="mixed")
    streams = dict(tuple(bank.sort_values("window_end", ascending=False).groupby(keys)))
    refs, counts = {}, {}
    for eid, row in meta.iterrows():
        stream = streams.get((row.participant_id, row.session_id))
        start = pd.to_datetime(float(row.start_ms), unit="ms", utc=True)
        chosen, limit = [], start
        if stream is not None:
            for idx, left, right in stream[["window_start", "window_end"]].itertuples():
                if right <= limit:
                    chosen.append(idx)
                    limit = left
                    if len(chosen) == 12:
                        break
        counts[eid] = len(chosen)
        refs[eid] = (stream.loc[chosen, HISTORY_FEATURES].median() if len(chosen) >= 3
                     else pd.Series(np.nan, index=HISTORY_FEATURES))
    ref = pd.DataFrame(refs).T.reindex(meta.index)
    return ref, pd.Series(counts).reindex(meta.index)


def load_v2_aux_pool(dataset_root):
    """Window v2 active_task (good/acceptable) dengan kode kesulitan protokol 0/1/2 untuk prior auxiliary."""
    v2_all = Path(dataset_root) / "v2_session_phase/window_15s_step_5s/all_data.csv"
    cols = ["window_id", "participant_id", "study_phase", "cognitive_load_label", "quality_tier"] + \
        CORE9 + HEAD4 + BASE1
    v2 = pd.read_csv(v2_all, usecols=cols)
    v2 = v2[v2.study_phase.eq("active_task") & v2.quality_tier.isin(["good", "acceptable"])
            & np.isfinite(v2[CORE9]).all(axis=1)].copy()
    v2["difficulty_code"] = v2.cognitive_load_label.map({"Low": 0, "Medium": 1, "High": 2})
    return v2.reset_index(drop=True)


# --------------------------------------------------------------------------------------
# Transformasi & bobot
# --------------------------------------------------------------------------------------
def robust_stats(values):
    med = np.nanmedian(values, axis=0)
    filled = np.where(np.isfinite(values), values, med)
    q75, q25 = np.percentile(filled, [75, 25], axis=0)
    iqr = np.where((q75 - q25) > 1e-9, q75 - q25, 1.0)
    return med, iqr


def robust_apply(values, med, iqr):
    values = np.where(np.isfinite(values), values, med)
    return np.clip((values - med) / iqr, -10.0, 10.0)


def global_robust(fit_vals, eval_vals):
    """asal: batch-07 exp03 `transform_windows(mode='global_robust')`."""
    med, iqr = robust_stats(fit_vals)
    return robust_apply(fit_vals, med, iqr), robust_apply(eval_vals, med, iqr)


def episode_balanced_window_weights(sub_df, y_bin, clip_max=5.0):
    """asal: batch-07 exp03. Bobot setara episode di dalam kelas, kelas setara total."""
    n = len(y_bin)
    w = np.ones(n, dtype=float)
    for cls in (0, 1):
        mask = y_bin == cls
        if not mask.any():
            continue
        g = sub_df.loc[mask]
        n_ep = g["episode_id"].nunique()
        ep_sz = g.groupby("episode_id")["window_id"].transform("count").to_numpy(dtype=float)
        wc = (n / (2.0 * n_ep)) / ep_sz
        if clip_max is not None:
            wc = np.minimum(wc, clip_max)
        wc = wc * ((n / 2.0) / wc.sum())
        w[mask] = wc
    return w


# --------------------------------------------------------------------------------------
# Metrik
# --------------------------------------------------------------------------------------
def safe_auc(y, s):
    y = np.asarray(y)
    return float(roc_auc_score(y, s)) if len(np.unique(y)) == 2 else np.nan


def confusion(y, pred):
    y, pred = np.asarray(y, int), np.asarray(pred, int)
    tp = int(((pred == 1) & (y == 1)).sum())
    fp = int(((pred == 1) & (y == 0)).sum())
    fn = int(((pred == 0) & (y == 1)).sum())
    tn = int(((pred == 0) & (y == 0)).sum())
    return tp, fp, fn, tn


def balanced_accuracy(y, pred):
    tp, fp, fn, tn = confusion(y, pred)
    rec = tp / (tp + fn) if tp + fn else 0.0
    spec = tn / (tn + fp) if tn + fp else 0.0
    return 0.5 * (rec + spec)


def within_person(y, score, pid):
    frame = pd.DataFrame({"y": np.asarray(y), "s": np.asarray(score), "p": np.asarray(pid)})
    aucs, conc, pairs = [], 0.0, 0
    for _, h in frame.groupby("p"):
        if h.y.nunique() == 2:
            aucs.append(roc_auc_score(h.y, h.s))
            pos, neg = h.s[h.y == 1].to_numpy(), h.s[h.y == 0].to_numpy()
            conc += (pos[:, None] > neg[None, :]).sum() + 0.5 * (pos[:, None] == neg[None, :]).sum()
            pairs += len(pos) * len(neg)
    return (float(np.mean(aucs)) if aucs else np.nan, len(aucs),
            float(conc / pairs) if pairs else np.nan, int(pairs))


def recall_at_fa(y, score, fa_cap=0.25):
    """Recall tertinggi dengan alarm palsu ≤ fa_cap (operating point pada kurva ROC)."""
    y, score = np.asarray(y, int), np.asarray(score, float)
    best = 0.0
    for thr in np.unique(score):
        pred = (score >= thr).astype(int)
        tp, fp, fn, tn = confusion(y, pred)
        if fp / max(fp + tn, 1) <= fa_cap:
            best = max(best, tp / max(tp + fn, 1))
    return best


def episode_metrics(y, score, pred, pid):
    y, score, pred = np.asarray(y, int), np.asarray(score, float), np.asarray(pred, int)
    tp, fp, fn, tn = confusion(y, pred)
    w_auc, w_n, w_pair, n_pairs = within_person(y, score, pid)
    return dict(episodes=len(y), participants=int(pd.Series(pid).nunique()), positives=int(y.sum()),
                majority_accuracy=float(1 - y.mean()), auc=safe_auc(y, score),
                pr_auc=float(average_precision_score(y, score)), balanced_accuracy=balanced_accuracy(y, pred),
                accuracy=float((y == pred).mean()), recall=tp / max(tp + fn, 1),
                false_alarm_rate=fp / max(fp + tn, 1), precision=tp / max(tp + fp, 1),
                recall_at_fa25=recall_at_fa(y, score, 0.25), within_person_auc=w_auc,
                within_person_people=w_n, within_person_pair_concordance=w_pair, within_person_pairs=n_pairs,
                tp=tp, fp=fp, fn=fn, tn=tn)


def cluster_bootstrap(frame, stat_fn, n_boot=2000, seed=SEED):
    """Bootstrap per peserta. `frame` minimal punya kolom participant_id."""
    rng = np.random.default_rng(seed)
    groups = {p: g for p, g in frame.groupby("participant_id")}
    ids = np.array(sorted(groups))
    vals = []
    for _ in range(n_boot):
        sample = pd.concat([groups[i] for i in rng.choice(ids, len(ids))], ignore_index=True)
        v = stat_fn(sample)
        if v is not None and np.isfinite(v):
            vals.append(v)
    vals = np.asarray(vals)
    return dict(lo=float(np.percentile(vals, 2.5)), hi=float(np.percentile(vals, 97.5)),
                p_le_0=float((vals <= 0).mean()), n=len(vals))


def logit_loglik(X, y):
    """Log-likelihood logistik tanpa regularisasi (untuk uji likelihood-ratio)."""
    model = LogisticRegression(C=1e6, max_iter=10000).fit(X, y)
    p = np.clip(model.predict_proba(X)[:, 1], 1e-12, 1 - 1e-12)
    return float(np.sum(y * np.log(p) + (1 - y) * np.log(1 - p))), p


def lr_test(base, extra, y):
    y = np.asarray(y, int)
    ll0, p0 = logit_loglik(base, y)
    ll1, p1 = logit_loglik(np.column_stack([base, extra]), y)
    stat = max(0.0, 2 * (ll1 - ll0))
    return dict(chi2=stat, p=float(chi2.sf(stat, 1)), auc_base=safe_auc(y, p0), auc_with_sensor=safe_auc(y, p1))


# --------------------------------------------------------------------------------------
# Ambang (asal: batch-07 exp03 `choose_thresholds_from_oof`, tanpa macro-F1)
# --------------------------------------------------------------------------------------
def choose_thresholds(y_true, probs):
    y_b, p_a = np.asarray(y_true, int), np.asarray(probs, float)
    chosen = {"fixed_050": 0.50}
    best_y, rec75, fa25 = None, [], []
    for thr in THRESHOLD_GRID:
        tp, fp, fn, tn = confusion(y_b, (p_a >= thr).astype(int))
        rec = tp / (tp + fn) if tp + fn else 0.0
        far = fp / (tn + fp) if tn + fp else 0.0
        bal = 0.5 * (rec + 1 - far)
        cand = (-bal, fp, abs(thr - 0.50), thr)
        if best_y is None or cand < best_y:
            best_y = cand
        if rec >= 0.75:
            rec75.append((fp, -tp, -thr, thr))
        if far <= 0.25:
            fa25.append((-tp, -bal, abs(thr - 0.50), thr))
    chosen["youden"] = float(best_y[3])
    chosen["recall75_min_fa"] = float(min(rec75)[3]) if rec75 else chosen["youden"]
    chosen["fa_cap25_max_rec"] = float(min(fa25)[3]) if fa25 else 0.65
    return chosen


class PlattBalanced:
    """Kalibrasi skor → probabilitas dengan bobot kelas seimbang (0,5 = batas seimbang)."""

    def fit(self, score, y):
        self.model = LogisticRegression(C=1e4, class_weight="balanced", max_iter=10000).fit(
            np.asarray(score, float).reshape(-1, 1), np.asarray(y, int))
        return self

    def transform(self, score):
        return self.model.predict_proba(np.asarray(score, float).reshape(-1, 1))[:, 1]


def aggregate(window_scores, episode_ids, how):
    s = pd.Series(np.asarray(window_scores, float), index=pd.Index(episode_ids, name="episode_id"))
    if how == "mean":
        return s.groupby(level=0).mean()
    if how == "median":
        return s.groupby(level=0).median()
    if how == "top50":
        return s.groupby(level=0).apply(lambda v: np.sort(v.to_numpy())[len(v) // 2:].mean())
    raise ValueError(how)


## Kode bersama · `shared/b8_engine.py`

In [3]:
"""Batch 08 · mesin pemodelan per-window → episode dan nested LOPO.

Bergantung pada `b8_common.py`. Diinline ke notebook setelah b8_common.
"""

if "b8_common_loaded" not in globals():
    from b8_common import *  # noqa: F401,F403

b8_engine_loaded = True

C_GRID = (0.05, 0.1, 0.25, 0.5, 1.0, 2.0)
AGGREGATIONS = ("mean", "median", "top50")
TARGETS = ("bin", "rating", "zrating")
NORMS = ("global", "hybrid")
AUX_OPTIONS = (False, True)
REFERENCE_CONFIG = dict(features="core9", norm="global", target="bin", aux=False, C=0.5, agg="mean")
CONFIG_FIELDS = ("features", "norm", "target", "aux", "C", "agg")


def config_key(cfg):
    return "|".join(f"{k}={cfg[k]}" for k in CONFIG_FIELDS)


def parse_key(key):
    cfg = dict(part.split("=", 1) for part in key.split("|"))
    cfg["aux"] = cfg["aux"] == "True"
    cfg["C"] = float(cfg["C"])
    return cfg


def complexity(cfg):
    """Tie-break: fitur lebih sedikit, tanpa tambahan, regularisasi lebih kuat lebih dulu."""
    return (len(FEATURE_SETS[cfg["features"]]), NORMS.index(cfg["norm"]), int(cfg["aux"]),
            TARGETS.index(cfg["target"]), cfg["C"], AGGREGATIONS.index(cfg["agg"]))


def all_configs(pool=None):
    pool = pool or {}
    return [dict(features=f, norm=n, target=t, aux=a, C=c, agg=g)
            for f in pool.get("features", tuple(FEATURE_SETS))
            for n in pool.get("norms", NORMS)
            for t in pool.get("targets", TARGETS)
            for a in pool.get("aux", AUX_OPTIONS)
            for c in pool.get("C", C_GRID)
            for g in pool.get("agg", AGGREGATIONS)]


def _design(fit_df, held_df, features, norm, refs, aux_block, track):
    cols = FEATURE_SETS[features]
    X_fit, X_held = global_robust(fit_df[cols].to_numpy(float), held_df[cols].to_numpy(float))
    blocks_fit, blocks_held = [X_fit], [X_held]
    if norm == "hybrid":
        hcols = [c for c in cols if c in HISTORY_FEATURES]
        ref_fit = refs.loc[fit_df.episode_id, hcols].to_numpy(float)
        ref_held = refs.loc[held_df.episode_id, hcols].to_numpy(float)
        fallback = np.nanmedian(fit_df[hcols].to_numpy(float), axis=0)
        ref_fit = np.where(np.isfinite(ref_fit), ref_fit, fallback)
        ref_held = np.where(np.isfinite(ref_held), ref_held, fallback)
        d_fit, d_held = global_robust(fit_df[hcols].to_numpy(float) - ref_fit,
                                      held_df[hcols].to_numpy(float) - ref_held)
        blocks_fit.append(d_fit)
        blocks_held.append(d_held)
    if aux_block is not None:
        blocks_fit.append(aux_block[0])
        blocks_held.append(aux_block[1])
    if track == "B":
        e_fit, e_held = global_robust(fit_df[[ELAPSED]].to_numpy(float), held_df[[ELAPSED]].to_numpy(float))
        blocks_fit.append(e_fit)
        blocks_held.append(e_held)
    return np.hstack(blocks_fit), np.hstack(blocks_held)


def _aux_scores(aux_pool, aux_people, fit_df, held_df, features):
    """Prior auxiliary: Ridge pada kode kesulitan protokol (0/1/2) dari window v2 aktif peserta yang diizinkan."""
    cols = FEATURE_SETS[features]
    pool = aux_pool[aux_pool.participant_id.isin(aux_people)]
    Xa = pool[cols].to_numpy(float)
    med, iqr = robust_stats(Xa)
    y = pool.difficulty_code.to_numpy(float)
    counts = pool.difficulty_code.map(pool.difficulty_code.value_counts()).to_numpy(float)
    model = Ridge(alpha=1.0).fit(robust_apply(Xa, med, iqr), y, sample_weight=len(y) / (3 * counts))
    s_fit = model.predict(robust_apply(fit_df[cols].to_numpy(float), med, iqr)).reshape(-1, 1)
    s_held = model.predict(robust_apply(held_df[cols].to_numpy(float), med, iqr)).reshape(-1, 1)
    smed, siqr = robust_stats(s_fit)
    return robust_apply(s_fit, smed, siqr), robust_apply(s_held, smed, siqr)


def score_fitset(df, refs, aux_pool, aux_extra_people, fit_people, held_people, configs, track="A",
                 return_windows=False, clip_max=5.0, label_column="y", rating_column="difficulty_rating"):
    """Latih semua konfigurasi pada `fit_people`; kembalikan skor episode (opsional window) untuk `held_people`."""
    fit_people, held_people = frozenset(fit_people), frozenset(held_people)
    if fit_people & held_people:
        raise ValueError("Peserta fitting dan held tumpang tindih.")
    fit_df = df[df.participant_id.isin(fit_people)].reset_index(drop=True)
    held_df = df[df.participant_id.isin(held_people)].reset_index(drop=True)
    if held_df.empty or fit_df[label_column].nunique() < 2:
        return None
    y_fit = fit_df[label_column].to_numpy(int)
    w_fit = episode_balanced_window_weights(fit_df, y_fit, clip_max=clip_max)
    rating = fit_df[rating_column].to_numpy(float)
    stats = fit_df.groupby("participant_id")[rating_column].agg(["mean", "std"])
    sd = stats["std"].where(stats["std"] > 0).fillna(1.0)
    z = ((fit_df[rating_column] - fit_df.participant_id.map(stats["mean"]))
         / fit_df.participant_id.map(sd)).to_numpy(float)
    aux_people = (fit_people | frozenset(aux_extra_people)) - held_people
    designs, aux_cache, windows, out = {}, {}, {}, {}
    for cfg in configs:
        dkey = (cfg["features"], cfg["norm"], cfg["aux"])
        if dkey not in designs:
            aux_block = None
            if cfg["aux"]:
                if cfg["features"] not in aux_cache:
                    aux_cache[cfg["features"]] = _aux_scores(aux_pool, aux_people, fit_df, held_df, cfg["features"])
                aux_block = aux_cache[cfg["features"]]
            designs[dkey] = _design(fit_df, held_df, cfg["features"], cfg["norm"], refs, aux_block, track)
        mkey = dkey + (cfg["target"], cfg["C"])
        if mkey not in windows:
            X_fit, X_held = designs[dkey]
            if cfg["target"] == "bin":
                model = LogisticRegression(C=cfg["C"], max_iter=5000, random_state=SEED)
                model.fit(X_fit, y_fit, sample_weight=w_fit)
                windows[mkey] = model.predict_proba(X_held)[:, 1]
            else:
                target = rating if cfg["target"] == "rating" else z
                windows[mkey] = Ridge(alpha=1.0 / cfg["C"]).fit(X_fit, target, sample_weight=w_fit).predict(X_held)
        out[config_key(cfg)] = aggregate(windows[mkey], held_df.episode_id, cfg["agg"])
    result = {"episode_scores": out}
    if return_windows:
        result["window_scores"] = {config_key(dict(zip(CONFIG_FIELDS, k[:2] + (k[3], k[2], k[4], "mean")))): v
                                   for k, v in windows.items()}
        result["held_windows"] = held_df[["window_id", "episode_id", "participant_id", "elapsed_active_s",
                                          label_column]].rename(columns={label_column: "y"})
    return result


def window_key(cfg):
    """Kunci skor window untuk konfigurasi (agregasi diabaikan)."""
    return config_key(dict(cfg, agg="mean"))


def select_config(inner_scores, inner_y, configs):
    """Pilih konfigurasi dari skor OOF inner-LOPO: AUC tertinggi → BalAcc (youden) → kompleksitas."""
    rows = []
    for cfg in configs:
        key = config_key(cfg)
        s = inner_scores[key].reindex(inner_y.index)
        auc = safe_auc(inner_y, s)
        p = PlattBalanced().fit(s, inner_y).transform(s)
        thr = choose_thresholds(inner_y, p)[PRIMARY_POLICY]
        bal = balanced_accuracy(inner_y, (p >= thr).astype(int))
        rows.append(dict(config=key, inner_auc=auc, inner_balacc=bal, complexity=complexity(cfg), **cfg))
    table = pd.DataFrame(rows).sort_values(["inner_auc", "inner_balacc", "complexity"],
                                           ascending=[False, False, True], kind="mergesort").reset_index(drop=True)
    table["rank"] = np.arange(1, len(table) + 1)
    return table


def run_fitsets(jobs, n_jobs=-1, verbose=0):
    from joblib import Parallel, delayed
    return Parallel(n_jobs=n_jobs, verbose=verbose)(delayed(score_fitset)(**job) for job in jobs)


def nested_lopo(df, meta, refs, aux_pool, aux_extra_people, configs, track="A", n_jobs=-1, verbose=0,
                clip_max=5.0, people=None, label_column="y", rating_column="difficulty_rating"):
    """Nested LOPO penuh: seleksi konfigurasi, kalibrasi, dan ambang hanya dari inner-LOPO tiap outer fold.

    `meta` harus berindeks episode_id dengan kolom participant_id dan `label_column`.
    """
    people = sorted(people or meta.participant_id.unique())
    common = dict(df=df, refs=refs, aux_pool=aux_pool, aux_extra_people=tuple(aux_extra_people), configs=configs,
                  track=track, clip_max=clip_max, label_column=label_column, rating_column=rating_column)
    jobs, index = [], []
    for p in people:
        rest = [q for q in people if q != p]
        jobs.append(dict(common, fit_people=rest, held_people=[p], return_windows=True))
        index.append(("outer", p, None))
        for q in rest:
            jobs.append(dict(common, fit_people=[r for r in rest if r != q], held_people=[q]))
            index.append(("inner", p, q))
    results = run_fitsets(jobs, n_jobs=n_jobs, verbose=verbose)
    outer, inner = {}, {}
    for (kind, p, _), r in zip(index, results):
        if r is None:
            continue
        if kind == "outer":
            outer[p] = r
        else:
            inner.setdefault(p, []).append(r["episode_scores"])
    keys = [config_key(c) for c in configs]
    rows, logs, all_outer, windows, selected_inner = [], [], [], [], {}
    for p in people:
        if p not in outer:
            continue
        inner_scores = {k: pd.concat([d[k] for d in inner[p]]) for k in keys}
        inner_y = meta.loc[inner_scores[keys[0]].index, label_column].astype(int)
        table = select_config(inner_scores, inner_y, configs)
        key = table.iloc[0].config
        s_in = inner_scores[key].reindex(inner_y.index)
        cal = PlattBalanced().fit(s_in, inner_y)
        thresholds = choose_thresholds(inner_y, cal.transform(s_in))
        outer_s = outer[p]["episode_scores"][key]
        prob = cal.transform(outer_s)
        held = meta.loc[outer_s.index]
        for pol, thr in thresholds.items():
            rows.append(pd.DataFrame({"episode_id": outer_s.index, "participant_id": held.participant_id.values,
                                      "y": held[label_column].values.astype(int), "fold": p, "config": key,
                                      "raw_score": outer_s.values, "prob": prob, "policy": pol, "threshold": thr,
                                      "predicted": (prob >= thr).astype(int)}))
        logs.append(table.head(20).assign(fold=p))
        for k in keys:
            s = outer[p]["episode_scores"][k]
            all_outer.append(pd.DataFrame({"episode_id": s.index, "fold": p, "config": k, "raw_score": s.values}))
        hw = outer[p]["held_windows"].copy()
        cfg = parse_key(key)
        raw_w = outer[p]["window_scores"][window_key(cfg)]
        hw["window_raw"] = raw_w
        hw["window_prob"] = cal.transform(raw_w)
        hw["fold"], hw["config"], hw["agg"] = p, key, cfg["agg"]
        hw["threshold_youden"] = thresholds[PRIMARY_POLICY]
        hw["cal_coef"] = float(cal.model.coef_[0, 0])
        hw["cal_intercept"] = float(cal.model.intercept_[0])
        windows.append(hw)
        selected_inner[p] = dict(key=key, inner=s_in, inner_y=inner_y, outer=outer_s,
                                 inner_all={k: inner_scores[k].reindex(inner_y.index) for k in keys},
                                 outer_all={k: outer[p]["episode_scores"][k] for k in keys},
                                 outer_windows=outer[p]["held_windows"],
                                 outer_window_scores=outer[p]["window_scores"], keys=keys)
    return dict(predictions=pd.concat(rows, ignore_index=True), selection_log=pd.concat(logs, ignore_index=True),
                all_outer=pd.concat(all_outer, ignore_index=True), windows=pd.concat(windows, ignore_index=True),
                selected_inner=selected_inner)


def stack_nested(sensor_res, context_res, meta, C=1.0):
    """Track B2: logistik 2-variabel pada skor sensor (Track A terpilih) + skor konteks, per outer fold.

    Stacker, kalibrasi, dan ambang dipelajari dari skor inner-LOPO kedua model (peserta outer tidak terlibat).
    """
    rows, coefs = [], []
    for p, s_info in sensor_res["selected_inner"].items():
        c_info = context_res["selected_inner"][p]
        idx = s_info["inner"].index.intersection(c_info["inner"].index)
        y_in = s_info["inner_y"].reindex(idx).astype(int)
        X_in = np.column_stack([s_info["inner"].reindex(idx), c_info["inner"].reindex(idx)])
        mu, sd = X_in.mean(axis=0), X_in.std(axis=0) + 1e-12
        model = LogisticRegression(C=C, class_weight="balanced", max_iter=5000).fit((X_in - mu) / sd, y_in)
        p_in = model.predict_proba((X_in - mu) / sd)[:, 1]
        thresholds = choose_thresholds(y_in, p_in)
        out_idx = s_info["outer"].index
        X_out = np.column_stack([s_info["outer"].reindex(out_idx), c_info["outer"].reindex(out_idx)])
        prob = model.predict_proba((X_out - mu) / sd)[:, 1]
        held = meta.loc[out_idx]
        for pol, thr in thresholds.items():
            rows.append(pd.DataFrame({"episode_id": out_idx, "participant_id": held.participant_id.values,
                                      "y": held.y.values.astype(int), "fold": p,
                                      "config": f"stack[{s_info['key']} + context]", "raw_score": prob, "prob": prob,
                                      "policy": pol, "threshold": thr, "predicted": (prob >= thr).astype(int)}))
        coefs.append(dict(fold=p, sensor_key=s_info["key"], coef_sensor=float(model.coef_[0, 0]),
                          coef_context=float(model.coef_[0, 1]), intercept=float(model.intercept_[0]),
                          mu_sensor=mu[0], sd_sensor=sd[0], mu_context=mu[1], sd_context=sd[1],
                          threshold_youden=thresholds[PRIMARY_POLICY]))
    return pd.concat(rows, ignore_index=True), pd.DataFrame(coefs)


def summarize_predictions(pred, meta, group_cols=("policy",), n_boot=2000):
    rows = []
    for keys, g in pred.groupby(list(group_cols)):
        keys = keys if isinstance(keys, tuple) else (keys,)
        m = episode_metrics(g.y, g.prob if "prob" in g else g.raw_score, g.predicted, g.participant_id)
        ci = cluster_bootstrap(g.assign(s=g.prob), lambda h: safe_auc(h.y, h.s), n_boot=n_boot)
        ci_b = cluster_bootstrap(g, lambda h: balanced_accuracy(h.y, h.predicted), n_boot=n_boot)
        rows.append(dict(zip(group_cols, keys), **m, auc_ci_lo=ci["lo"], auc_ci_hi=ci["hi"],
                         balacc_ci_lo=ci_b["lo"], balacc_ci_hi=ci_b["hi"]))
    return pd.DataFrame(rows)


def paired_delta(new, ref, n_boot=2000):
    """Bootstrap berpasangan per peserta: Δ AUC dan Δ BalAcc (new − ref) pada episode yang sama."""
    m = new[["episode_id", "participant_id", "y", "prob", "predicted"]].merge(
        ref[["episode_id", "prob", "predicted"]], on="episode_id", suffixes=("_new", "_ref"), validate="1:1")
    d_auc = safe_auc(m.y, m.prob_new) - safe_auc(m.y, m.prob_ref)
    d_bal = balanced_accuracy(m.y, m.predicted_new) - balanced_accuracy(m.y, m.predicted_ref)
    ci_auc = cluster_bootstrap(m, lambda h: safe_auc(h.y, h.prob_new) - safe_auc(h.y, h.prob_ref), n_boot)
    ci_bal = cluster_bootstrap(m, lambda h: balanced_accuracy(h.y, h.predicted_new)
                               - balanced_accuracy(h.y, h.predicted_ref), n_boot)
    return dict(episodes=len(m), delta_auc=d_auc, delta_auc_lo=ci_auc["lo"], delta_auc_hi=ci_auc["hi"],
                p_delta_auc_le_0=ci_auc["p_le_0"], delta_balacc=d_bal, delta_balacc_lo=ci_bal["lo"],
                delta_balacc_hi=ci_bal["hi"], p_delta_balacc_le_0=ci_bal["p_le_0"])


CONTEXT_CONFIG = dict(features="core9", norm="global", target="bin", aux=False, C=1.0, agg="mean")


def context_frame(df):
    """Salinan window dengan fitur sensor dinolkan: desain Track B hanya berisi log waktu aktif berjalan."""
    ctx = df.copy()
    for col in CORE9:
        ctx[col] = 0.0
    return ctx


def context_nested(df, meta, n_jobs=-1):
    return nested_lopo(context_frame(df), meta, None, None, (), [dict(CONTEXT_CONFIG)], track="B", n_jobs=n_jobs)



def ensemble_nested(res, meta, label="ensemble_all_configs"):
    """Rata-rata skor seluruh konfigurasi pool (tanpa seleksi). Setiap skor distandardisasi dengan statistik inner-OOF
    fold tersebut; kalibrasi dan ambang dari inner-OOF. Menghilangkan variansi pemilihan konfigurasi."""
    rows, windows = [], []
    for p, info in res["selected_inner"].items():
        keys, y_in = info["keys"], info["inner_y"]
        stats = {k: (info["inner_all"][k].mean(), info["inner_all"][k].std() + 1e-12) for k in keys}
        s_in = sum((info["inner_all"][k] - stats[k][0]) / stats[k][1] for k in keys) / len(keys)
        s_out = sum((info["outer_all"][k] - stats[k][0]) / stats[k][1] for k in keys) / len(keys)
        cal = PlattBalanced().fit(s_in, y_in)
        thresholds = choose_thresholds(y_in, cal.transform(s_in))
        prob = cal.transform(s_out)
        held = meta.loc[s_out.index]
        for pol, thr in thresholds.items():
            rows.append(pd.DataFrame({"episode_id": s_out.index, "participant_id": held.participant_id.values,
                                      "y": held.y.values.astype(int), "fold": p, "config": label,
                                      "raw_score": s_out.values, "prob": prob, "policy": pol, "threshold": thr,
                                      "predicted": (prob >= thr).astype(int)}))
        # Skor window per konfigurasi → agregasi berjalan dilakukan di exp03 dengan statistik yang sama.
        hw = info["outer_windows"].copy()
        for k in keys:
            hw["w|" + k] = info["outer_window_scores"][window_key(parse_key(k))]
        hw["fold"] = p
        hw["cal_coef"], hw["cal_intercept"] = float(cal.model.coef_[0, 0]), float(cal.model.intercept_[0])
        hw["threshold_youden"] = thresholds[PRIMARY_POLICY]
        for k in keys:
            hw["mu|" + k], hw["sd|" + k] = stats[k]
        windows.append(hw)
        info_e = dict(key=label, inner=s_in, inner_y=y_in, outer=s_out)
        res.setdefault("ensemble_inner", {})[p] = info_e
    return pd.concat(rows, ignore_index=True), pd.concat(windows, ignore_index=True)


def stack_from_ensemble(res, context_res, meta):
    """Track B3: stacking skor ensemble Track A + skor konteks (sama seperti stack_nested)."""
    proxy = {"selected_inner": res["ensemble_inner"]}
    return stack_nested(proxy, context_res, meta)


## Kode bersama · `shared/b8_v3_labels.py`

In [4]:
"""Batch 08 · label V3 High vs Non-High per fold (kebijakan v3_high_vs_rest_foldwise_v1).

Fungsi `make_reference`, `reference_for`, `label_episode`, dan `fold_labels` disalin verbatim dari kode
batch-06 v3 exp13 (diinline di models/batch-07/exp01/EyeHeartCognitiveLoad.ipynb sel 5) agar rubrik V3
identik. Label dihitung ulang dari peserta fitting pada setiap fold; jangan dibekukan ke file input.
"""

from collections import defaultdict

import numpy as np
import pandas as pd

b8_v3_labels_loaded = True
LEGACY_POLICY = 'behavioral_concordance_v1'
POLICY = 'behavioral_concordance_v2'
MIN_REFERENCE = 5


def make_reference(episodes, splits, *, policy=POLICY):
    """One first-exposure observation per person/item; only successful Train cases."""
    seen, items = set(), defaultdict(list)
    for row in sorted(episodes, key=lambda r: (r['start_ms'], r['episode_id'])):
        identity = row['participant_id'], row['item_key']
        if identity in seen:
            continue
        seen.add(identity)
        if (splits[row['participant_id']] == 'Train' and row['valid'] and row['completed']
                and (policy == LEGACY_POLICY or row.get('reference_eligible', True))
                and row['success_submission_type'] == 'manual' and row['active_seconds'] > 0):
            items[row['item_key']].append({'participant_id': row['participant_id'],
                'episode_id': row['episode_id'], 'active_seconds': row['active_seconds']})
    return {'policy': policy, 'minimum_other_people': MIN_REFERENCE, 'quantiles': [1/3,2/3],
            'quantile_method': 'linear', 'source_split': 'Train', 'self_reference': False,
            'items': {key: sorted(values, key=lambda v: v['participant_id']) for key, values in sorted(items.items())}}


def reference_for(reference, episode):
    values = [v for v in reference['items'].get(episode['item_key'], [])
              if v['participant_id'] != episode['participant_id']]
    result = {'participants': [v['participant_id'] for v in values], 'count': len(values),
                  'lower_seconds': None, 'upper_seconds': None, 'reason': ''}
    if len(values) < MIN_REFERENCE:
        result['reason'] = 'insufficient_reference'
    else:
        lower, upper = np.quantile([v['active_seconds'] for v in values], [1/3,2/3], method='linear')
        result.update(lower_seconds=float(lower), upper_seconds=float(upper))
        if not 0 < lower < upper:
            result['reason'] = 'degenerate_reference'
    return result


def label_episode(episode, reference):
    limits = reference_for(reference, episode)
    result = {'cognitive_load_label': '', 'label_reason': '', 'time_band': '', 'performance_band': '',
                  'reference_count': limits['count'], 'reference_participants': limits['participants'],
                  'lower_seconds': limits['lower_seconds'], 'upper_seconds': limits['upper_seconds']}
    if not episode['valid']:
        result['label_reason'] = 'invalid_event_trace'
    elif limits['reason']:
        result['label_reason'] = limits['reason']
    else:
        seconds, errors = episode['active_seconds'], episode['manual_errors']
        time_band = ('Low' if seconds <= limits['lower_seconds'] else
                     'Medium' if seconds <= limits['upper_seconds'] else 'High')
        performance = ('Low' if episode['completed'] and errors == 0 and episode['manual_submissions'] == 1
                       and episode['success_submission_type'] == 'manual' else
                       'Medium' if episode['completed'] and 1 <= errors <= 2 else 'High' if errors >= 3 else '')
        result.update(time_band=time_band, performance_band=performance)
        # An unfinished duration is only a lower bound: never use it for Low/Medium.
        if performance == time_band and (episode['completed'] or time_band == 'High'):
            result.update(cognitive_load_label=time_band, label_reason='concordant_behavioral_evidence')
        else:
            result['label_reason'] = 'conflicting_or_insufficient_behavior'
    return result


def fold_labels(records, fit_people):
    roles = {row["participant_id"]: ("Train" if row["participant_id"] in fit_people else "held")
             for row in records}
    reference = make_reference(records, roles)
    return pd.Series({r["episode_id"]: label_episode(r, reference)["cognitive_load_label"]
                      for r in records})


def binary_high(labels):
    """Low/Medium → 0, High → 1; episode tanpa label tetap kosong."""
    return labels.map({"High": 1, "Medium": 0, "Low": 0})


## Eksperimen

In [5]:
"""Batch 08 · Exp04 · Validasi silang V3 (High vs Non-High per fold) dan 40 episode irisan V3∩V4.

1. Pipeline per-window Track A (konfigurasi yang paling sering terpilih di exp02) dilatih pada label V3 yang
   dihitung ulang per fold. Diperiksa apakah skor masih mengikuti durasi di dalam label yang sama (masalah V3 batch 07).
2. Skor outer V4 Track A dari exp02 dinilai pada 40 episode irisan dengan keempat target exp04 batch 07.
"""

if "b8_engine_loaded" not in globals():
    import sys
    from pathlib import Path
    sys.path.insert(0, str(Path(__file__).resolve().parents[1] / "shared"))
    from b8_common import *  # noqa: F401,F403
    from b8_engine import *  # noqa: F401,F403
    from b8_v3_labels import *  # noqa: F401,F403

BATCH07_V3 = {"Train_LOPO_svm_rbf_center_only": 0.768, "validation_svm_rbf_center_only": 0.750,
              "Test_historical_svm_rbf_center_only": 0.500}
BATCH07_INTERSECT_V4_CORE9 = {"behavioral_high_v3": 0.7116, "self_report_7_10_v4": 0.7742,
                              "concordant_both_high": 0.7879, "either_high_union": 0.7208}


def load_v3(dataset_root):
    pkg = Path(dataset_root) / "v3_behavioral_proxy"
    hashes = {f"v3/{r}": sha256(pkg / r) for r in ("window_15s_step_5s/good_acceptable.csv",
                                                     "window_15s_step_5s/window_assignments.csv",
                                                     "episode_labels.csv", "participant_splits.csv")}
    win = pd.read_csv(pkg / "window_15s_step_5s/good_acceptable.csv")
    wa = pd.read_csv(pkg / "window_15s_step_5s/window_assignments.csv")[["window_id", "episode_id", "split"]]
    win = win.merge(wa, on="window_id", validate="1:1")
    win["window_start_dt"] = pd.to_datetime(win.window_start, utc=True, format="mixed")
    win = win.sort_values(["session_id", "window_start_dt", "window_id"]).reset_index(drop=True)
    ev = pd.read_csv(pkg / "episode_labels.csv", keep_default_na=False)
    ev["intervals"] = ev.intervals.map(json.loads)
    records = ev.to_dict("records")
    splits = pd.read_csv(pkg / "participant_splits.csv").set_index("participant_id").split
    ep = ev.set_index("episode_id")
    win["difficulty_rating"] = 0.0  # tidak dipakai (target biner)
    return win, records, splits, ep, hashes


def v3_fold(win, records, refs, aux, fit_people, held_people, configs):
    labels = binary_high(fold_labels(records, frozenset(fit_people)).replace("", np.nan))
    d = win.assign(y=win.episode_id.map(labels)).dropna(subset=["y"])
    d["y"] = d.y.astype(int)
    r = score_fitset(d, refs, aux, (), fit_people, held_people, configs, track="A")
    if r is None:
        return None
    scores = pd.DataFrame(r["episode_scores"])
    scores["y"] = labels.reindex(scores.index).values.astype(int)
    return scores


def v3_protocol(win, records, refs, aux, ep, configs, outer_people, fit_pool, protocol):
    """LOPO pada label V3 per fold; skor = rata-rata z seluruh konfigurasi pool (seperti Track A-ens)."""
    from joblib import Parallel, delayed
    keys = [config_key(c) for c in configs]
    rows = []
    for p in outer_people:
        held = [p] if isinstance(p, str) else list(p)
        fit_people = [q for q in fit_pool if q not in held]
        outer = v3_fold(win, records, refs, aux, fit_people, held, configs)
        if outer is None:
            continue
        inner = Parallel(n_jobs=-1)(delayed(v3_fold)(win, records, refs, aux, [r for r in fit_people if r != q], [q],
                                                     configs) for q in fit_people)
        inner = pd.concat([i for i in inner if i is not None])
        stats = {k: (inner[k].mean(), inner[k].std() + 1e-12) for k in keys}
        s_in = sum((inner[k] - stats[k][0]) / stats[k][1] for k in keys) / len(keys)
        s_out = sum((outer[k] - stats[k][0]) / stats[k][1] for k in keys) / len(keys)
        cal = PlattBalanced().fit(s_in, inner.y)
        thr = choose_thresholds(inner.y, cal.transform(s_in))[PRIMARY_POLICY]
        rows.append(pd.DataFrame({"episode_id": outer.index, "score": s_out.values, "y": outer.y.values,
                                  "prob": cal.transform(s_out), "threshold": thr, "fold": str(p)}))
    pred = pd.concat(rows, ignore_index=True)
    pred["predicted"] = (pred.prob >= pred.threshold).astype(int)
    pred["participant_id"] = pred.episode_id.map(ep.participant_id)
    pred["active_seconds"] = pred.episode_id.map(ep.active_seconds.astype(float))
    pred["protocol"] = protocol
    return pred


def run(dataset_root, output_root, exp02_dir, n_jobs=-1):
    win, records, splits, ep, hashes = load_v3(dataset_root)
    exp02_manifest = json.loads((Path(exp02_dir) / "manifest.json").read_text(encoding="utf-8"))
    pool = exp02_manifest["pool"]
    pool = {k: tuple(v) for k, v in pool.items()} if isinstance(pool, dict) else None
    # V3 tidak punya rating 1–10 → hanya konfigurasi target biner dari pool exp02.
    configs = [c for c in all_configs(pool) if c["target"] == "bin"]
    meta_v3 = ep[["participant_id", "session_id", "start_ms"]].copy()
    refs, _ = history_reference(dataset_root, win, meta_v3)
    aux = load_v2_aux_pool(dataset_root)
    cfg = {"ensemble_of": len(configs), "pool": pool}
    train = sorted(splits[splits.eq("Train")].index)
    val = sorted(splits[splits.eq("validation")].index)
    test = sorted(splits[splits.eq("Test")].index)
    train_w = [p for p in train if p in set(win.participant_id)]
    preds = [v3_protocol(win, records, refs, aux, ep, configs, train_w, train_w, "Train_LOPO"),
             v3_protocol(win, records, refs, aux, ep, configs, [tuple(val)], train_w, "validation"),
             v3_protocol(win, records, refs, aux, ep, configs, [tuple(test)], train_w, "Test_historical")]
    pred = pd.concat(preds, ignore_index=True)

    rows = []
    for proto, g in pred.groupby("protocol"):
        m = episode_metrics(g.y, g.prob, g.predicted, g.participant_id)
        non = g[g.y == 0]
        rho_non = spearmanr(non.prob, non.active_seconds)[0] if len(non) > 2 else np.nan
        rows.append(dict(protocol=proto, config=f"ensemble_{len(configs)}_configs", **m, spearman_prob_vs_duration_nonhigh=rho_non,
                         spearman_prob_vs_duration_all=spearmanr(g.prob, g.active_seconds)[0],
                         batch07_auc_svm=BATCH07_V3.get(f"{proto}_svm_rbf_center_only")))
    v3m = pd.DataFrame(rows)
    tr = pred[pred.protocol.eq("Train_LOPO")]
    dur_test = lr_test(np.log(tr[["active_seconds"]].to_numpy()), logit(np.clip(tr.prob, 1e-6, 1 - 1e-6)), tr.y)

    # 40 episode irisan (definisi target dari exp04 batch 07)
    b7 = sorted(Path(dataset_root).parents[2].glob("models/batch-07/exp04/outputs/*/episode_predictions.csv"))[-1]
    inter = pd.read_csv(b7).drop_duplicates("episode_id")[["episode_id", "participant_id", "y_v3_high",
                                                            "y_v4_overload"]]
    inter["concordant_both_high"] = (inter.y_v3_high & inter.y_v4_overload).astype(int)
    inter["either_high_union"] = (inter.y_v3_high | inter.y_v4_overload).astype(int)
    a = pd.read_csv(Path(exp02_dir) / "trackA_ens_outer_predictions.csv")
    a = a[a.policy.eq(PRIMARY_POLICY)].set_index("episode_id").prob
    inter["trackA_prob"] = inter.episode_id.map(a)
    irows = []
    for target, col in [("behavioral_high_v3", "y_v3_high"), ("self_report_7_10_v4", "y_v4_overload"),
                        ("concordant_both_high", "concordant_both_high"), ("either_high_union", "either_high_union")]:
        g = inter.dropna(subset=["trackA_prob"])
        ci = cluster_bootstrap(g.assign(y=g[col]), lambda h: safe_auc(h.y, h.trackA_prob))
        irows.append(dict(target=target, episodes=len(g), positives=int(g[col].sum()),
                          auc_trackA=safe_auc(g[col], g.trackA_prob), auc_ci_lo=ci["lo"], auc_ci_hi=ci["hi"],
                          batch07_v4_logreg_core9=BATCH07_INTERSECT_V4_CORE9[target]))
    im = pd.DataFrame(irows)

    out = new_run_dir(output_root, "v3check")
    pred.to_csv(out / "v3_episode_predictions.csv", index=False)
    v3m.to_csv(out / "v3_metrics.csv", index=False)
    im.to_csv(out / "intersect40_metrics.csv", index=False)
    pd.DataFrame([dict(test="v3_score_over_log_duration_train_lopo", **dur_test)]).to_csv(
        out / "v3_duration_test.csv", index=False)
    save_json(out / "manifest.json", {"inputs": hashes, "label_policy": V3_LABEL_POLICY, "config": cfg,
                                      "exp02_run": str(exp02_dir), "batch07_intersect_source": str(b7),
                                      "environment": environment_info()})
    print(v3m[["protocol", "episodes", "positives", "auc", "balanced_accuracy", "majority_accuracy",
               "spearman_prob_vs_duration_nonhigh", "batch07_auc_svm"]].to_string(index=False))
    print("Skor V3 di atas log durasi (Train LOPO):", dur_test)
    print(im.to_string(index=False))
    print("ZIP:", zip_run(out))
    return out


## Jalankan dan simpan output

In [6]:
RUN_DIR = run(DATASET_ROOT, OUTPUT_ROOT, latest_run(BATCH_ROOT / 'exp02_nested_main/outputs', 'nested'))
print('Output:', RUN_DIR)

       protocol  episodes  positives      auc  balanced_accuracy  majority_accuracy  spearman_prob_vs_duration_nonhigh  batch07_auc_svm
Test_historical        18          2 0.656250           0.593750           0.888889                           0.117647            0.500
     Train_LOPO        37         16 0.431548           0.541667           0.567568                           0.240260            0.768
     validation        10          3 0.095238           0.500000           0.700000                          -0.214286            0.750
Skor V3 di atas log durasi (Train LOPO): {'chi2': 0.3323846291901571, 'p': 0.5642582943612727, 'auc_base': 0.9761904761904762, 'auc_with_sensor': 0.9761904761904762}
              target  episodes  positives  auc_trackA  auc_ci_lo  auc_ci_hi  batch07_v4_logreg_core9
  behavioral_high_v3        40         11    0.783699   0.547584   0.949226                   0.7116
 self_report_7_10_v4        40          9    0.928315   0.813014   1.000000             